In [1]:
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

try:
    client = MongoClient('mongodb://bigdata_mongodb:27017/')
    db = client['proyecto_bigdata']
    coleccion = db['alojamientos']
    print("Conexion a MongoDB exitosa!")
except Exception as e:
    print(f"Error conectando a MongoDB: {e}")

Conexion a MongoDB exitosa!


In [5]:
# Ciudades a scrapear (Norte, Centro, Sur de Chile)
ciudades = [
    'Santiago', 'Valparaiso', 'Vina del Mar',  # Centro
    'Arica', 'Iquique', 'Antofagasta',           # Norte
    'Puerto Montt', 'Pucon', 'Puerto Varas'      # Sur
]

def limpiar_precio(texto):
    """Extrae solo numeros del texto del precio"""
    numeros = ''
    for c in texto:
        if c.isdigit():
            numeros += c
    if not numeros:
        return None
    precio = float(numeros)
    # filtro: precios razonables para Chile (CLP)
    if precio < 10000 or precio > 5000000:
        return None
    return precio

def obtener_estrellas(hotel):
    """Lee la cantidad de estrellas del hotel"""
    try:
        # estrella completa
        stars = hotel.find_elements(
            By.CSS_SELECTOR, 
            '[data-testid="rating-stars"] span.fc70cba028.bdc459fcb4.f24706dc71:not(.e2cec97860)'
        )
        if stars:
            return len(stars)
        # Alternativa: buscar aria-label con "X de 5"
        star_div = hotel.find_element(By.CSS_SELECTOR, '[data-testid="rating-stars"]')
        parent = star_div.find_element(By.XPATH, '..')
        label = parent.get_attribute('aria-label')
        if label and 'de 5' in label:
            return int(label.split(' de 5')[0].strip())
    except:
        pass
    return 0

def obtener_tipo(hotel):
    """Lee el tipo de alojamiento"""
    try:
        
        desc = hotel.find_element(By.CSS_SELECTOR, '.fff1944c52').text.lower()
        if 'hotel' in desc:
            return 'hotel'
        elif 'apart' in desc or 'apartamento' in desc:
            return 'apartamento'
        elif 'hostal' in desc or 'hostel' in desc:
            return 'hostal'
        elif 'cabaña' in desc or 'cabana' in desc:
            return 'cabana'
        else:
            return 'hotel'
    except:
        return 'hotel'

def es_solo_hotel(hotel):
    """Filtra para quedarse solo con hoteles, no cabañas ni apartamentos"""
    try:
        desc = hotel.find_element(By.CSS_SELECTOR, '.fff1944c52').text.lower()
        palabras_excluir = ['cabaña', 'cabana', 'domo', 'hostal', 
                           'hostel', 'apart', 'lodge', 'camping']
        for palabra in palabras_excluir:
            if palabra in desc:
                return False
        nombre = hotel.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text.lower()
        for palabra in palabras_excluir:
            if palabra in nombre:
                return False
        return True
    except:
        return True

def configurar_driver():
    """Configura Chrome con opciones anti-deteccion"""
    opciones = Options()
    opciones.add_argument('--no-sandbox')
    opciones.add_argument('--disable-dev-shm-usage')
    opciones.add_argument('--disable-blink-features=AutomationControlled')
    opciones.add_experimental_option('excludeSwitches', ['enable-automation'])
    opciones.add_experimental_option('useAutomationExtension', False)
    opciones.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=opciones
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver

def scraper_booking(ciudad):
    """Scraper principal para una ciudad"""
    
    # URL con:
    # - Habitacion matrimonial 
    # - Solo hoteles (nflt=ht_id%3D204 = filtro tipo Hotel)
    # - Fechas fijas 3 noches para que aparezcan precios
    # - Idioma espanol 
    url = (
        f'https://www.booking.com/searchresults.es-ar.html?'
        f'ss={ciudad.replace(" ", "+")}%2C+Chile'
        f'&checkin=2025-06-01'
        f'&checkout=2025-06-04'
        f'&group_adults=2'
        f'&no_rooms=1'
        f'&group_children=0'
        f'&nflt=ht_id%3D204'  # Filtro: solo hoteles
        f'&order=popularity'
    )
    
    print(f'\n{"="*50}')
    print(f'Ciudad: {ciudad}')
    print(f'{"="*50}')
    
    driver = None
    try:
        driver = configurar_driver()
        driver.get(url)
        time.sleep(6)
        
        print('\n>>> ACCION REQUERIDA <<<')
        print('1. Abre: localhost:6080/vnc.html')
        print('2. Verifica que ves HOTELES con PRECIOS')
        print('3. Si hay captcha, resuelvelo')
        print('4. Cuando veas precios visibles, vuelve aqui')
        input('>>> Presiona ENTER para comenzar a extraer datos <<<\n')
        
        time.sleep(2)
        
        hoteles = driver.find_elements(
            By.CSS_SELECTOR, '[data-testid="property-card"]'
        )
        
        if not hoteles:
            print(f'Sin resultados para {ciudad}')
            return 0
        
        print(f'Hoteles encontrados: {len(hoteles)}')
        guardados = 0
        omitidos = 0
        sin_precio = 0
        
        for i, hotel in enumerate(hoteles):
            try:
                
                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'center'});", hotel
                )
                time.sleep(0.8)
                
                # Obtener nombre
                try:
                    nombre = hotel.find_element(
                        By.CSS_SELECTOR, '[data-testid="title"]'
                    ).text.strip()
                except:
                    continue
                
                if not nombre:
                    continue
                
                # Filtrar: solo hoteles
                if not es_solo_hotel(hotel):
                    omitidos += 1
                    print(f'  [{i+1}] OMITIDO (no es hotel): {nombre[:35]}')
                    continue
                
                # Intentar click en "Mostrar precios"
                try:
                    btn = hotel.find_element(
                        By.XPATH, 
                        './/span[contains(text(),"Mostrar precios")]/..'
                    )
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(2)
                except:
                    pass
                
                # Extraer precio con multiples selectores
                precio = None
                selectores_precio = [
                    '[data-testid="price-and-discounted-price"]',
                    '[data-testid="price"]',
                    '.prco-valign__middle-helper',
                    '[data-testid="availability-rate-information"]',
                ]
                for selector in selectores_precio:
                    try:
                        elem = hotel.find_element(By.CSS_SELECTOR, selector)
                        texto = elem.text.strip()
                        if texto:
                            precio = limpiar_precio(texto)
                            if precio:
                                break
                    except:
                        continue
                
                if not precio:
                    sin_precio += 1
                    print(f'  [{i+1}] SIN PRECIO: {nombre[:35]}')
                    # Guardamos igual pero con precio 0 para no perder el registro
                    precio = 0.0
                else:
                    print(f'  [{i+1}] ${precio:,.0f} | {nombre[:35]}')
                
                # Extraer puntuacion
                puntuacion = None
                try:
                    punt_elem = hotel.find_element(
                        By.CSS_SELECTOR, '[data-testid="review-score"]'
                    )
                    punt_texto = punt_elem.text.strip()
                    for palabra in punt_texto.replace('\n', ' ').split():
                        try:
                            val = float(palabra.replace(',', '.'))
                            if 1.0 <= val <= 10.0:
                                puntuacion = val
                                break
                        except:
                            continue
                except:
                    puntuacion = None
                
                # Extraer estrellas
                estrellas = obtener_estrellas(hotel)
                
                # Extraer tipo de alojamiento
                tipo = obtener_tipo(hotel)
                
                # Extraer URL
                try:
                    url_hotel = hotel.find_element(
                        By.CSS_SELECTOR, '[data-testid="title-link"]'
                    ).get_attribute('href')
                    # Limpiar URL (quitar parametros de tracking)
                    url_hotel = url_hotel.split('?')[0] if '?' in url_hotel else url_hotel
                except:
                    url_hotel = url
                
                # Determinar zona geografica
                if ciudad in ['Arica', 'Iquique', 'Antofagasta']:
                    zona = 'Norte'
                elif ciudad in ['Santiago', 'Valparaiso', 'Vina del Mar']:
                    zona = 'Centro'
                else:
                    zona = 'Sur'
                
                # Armar registro
                registro = {
                    'nombre_hotel': nombre,
                    'precio_noche': precio,
                    'ciudad': ciudad,
                    'zona_geografica': zona,
                    'estrellas': estrellas,
                    'tipo_alojamiento': tipo,
                    'tipo_habitacion': 'matrimonial',
                    'adultos': 2,
                    'noches': 3,
                    'puntuacion': puntuacion,
                    'fecha_captura': datetime.now(),
                    'url_origen': url_hotel,
                    'plataforma': 'Booking.com',
                    'integrante': 'Camila-rojas',
                    'grupo': 'G5_Turismo_Hoteleria'
                }
                
                # Guardar o actualizar en MongoDB
                coleccion.update_one(
                    {
                        'nombre_hotel': nombre,
                        'ciudad': ciudad,
                        'plataforma': 'Booking.com'
                    },
                    {'$set': registro},
                    upsert=True
                )
                guardados += 1
                
            except Exception as e:
                print(f'  Error hotel {i+1}: {str(e)[:50]}')
                continue
        
        print(f'\nResumen {ciudad}:')
        print(f'  Guardados:  {guardados}')
        print(f'  Sin precio: {sin_precio}')
        print(f'  Omitidos:   {omitidos}')
        return guardados
        
    except Exception as e:
        print(f'Error general en {ciudad}: {e}')
        return 0
    finally:
        if driver:
            driver.quit()


# EJECUCION PRINCIPAL

total_antes = coleccion.count_documents({'plataforma': 'Booking.com'})
print(f'Registros Booking en MongoDB antes: {total_antes}')
print(f'Ciudades a procesar: {len(ciudades)}')

total_nuevos = 0

for ciudad in ciudades:
    nuevos = scraper_booking(ciudad)
    total_nuevos += nuevos
    
    if ciudad != ciudades[-1]:  # No esperar en la ultima
        print(f'\nEsperando 15 segundos...')
        time.sleep(15)

# Resumen final
total_despues = coleccion.count_documents({'plataforma': 'Booking.com'})
print(f'\n{"="*50}')
print(f'SCRAPING COMPLETADO')
print(f'{"="*50}')
print(f'Registros antes:  {total_antes}')
print(f'Registros ahora:  {total_despues}')
print(f'Nuevos/actualizados: {total_despues - total_antes}')
print(f'{"="*50}')

Registros Booking en MongoDB antes: 0
Ciudades a procesar: 9

Ciudad: Santiago

>>> ACCION REQUERIDA <<<
1. Abre: localhost:6080/vnc.html
2. Verifica que ves HOTELES con PRECIOS
3. Si hay captcha, resuelvelo
4. Cuando veas precios visibles, vuelve aqui


>>> Presiona ENTER para comenzar a extraer datos <<<
 


Hoteles encontrados: 26
  [1] $104,428 | Hotel Boutique Holitel
  [2] $596,854 | Renaissance Santiago by Marriott
  [3] $157,481 | Hotel Tempo Rent
  [4] $216,362 | Hotel Nodo - El primer hotel de neg
  [5] $85,609 | Hotel El Rosedal
  [6] $112,599 | Hotel HW EXPRESS
  [7] $138,916 | Hotel Capital Bellet
  [8] $97,008 | Mito Casa Hotel
  [9] $119,423 | Hotel Canciller - Ex Hotel Presiden
  [10] $96,764 | Infinity Park Hotel
  [11] $168,636 | Hotel Diego de Velazquez
  [12] $113,737 | Hotel Montecarlo Santiago
  [13] $99,738 | Hotel Brasilia
  [14] $114,699 | Hotel HW Libertad
  [15] $150,517 | Tagle Hotel Boutique
  [16] $164,568 | Park Plaza Santiago
  [17] OMITIDO (no es hotel): One Nk Apartments
  [18] OMITIDO (no es hotel): Apart Hotel Viva Providencia
  [19] $137,534 | Hotel Altiplanico Bellas Artes
  [20] $165,355 | hUB Providencia
  [21] $211,000 | Novotel Santiago Vitacura
  [22] $49,362 | Hotel Guayaquil
  [23] $115,320 | Nest Paris-Londres
  [24] OMITIDO (no es hotel): Apart 

>>> Presiona ENTER para comenzar a extraer datos <<<
 


Hoteles encontrados: 25
  [1] $199,056 | Fauna Hotel
  [2] $174,708 | Casablu Hotel
  [3] $122,485 | Hotel Brighton
  [4] $159,809 | Hotel Winebox Valparaiso
  [5] $152,232 | Hotel Diego de Almagro Valparaíso
  [6] $331,621 | Hotel Casa Higueras
  [7] $175,137 | Casa Galos Hotel & Lofts
  [8] $125,985 | Casa California Guesthouse
  [9] $209,975 | Hotel Casa Somerscales
  [10] $142,000 | ibis Valparaiso
  [11] $294,490 | Palacio Astoreca
  [12] $165,180 | BO Hotel & Terraza
  [13] $198,426 | Hotel Boutique Acontraluz
  [14] $244,149 | AYCA La Flora Hotel Boutique
  [15] $157,901 | Hotel Faro Azul Valparaíso Cerro Al
  [16] $148,732 | Hotel boutique Paihuen
  [17] $236,222 | Casa Puente Hotel Boutique
  [18] $105,924 | Hotel Reina Victoria Valparaíso
  [19] $84,253 | Casa Esmeralda
  [20] $72,459 | Comarca Valparaíso
  [21] $166,580 | New Voga Hotel Boutique
  [22] $201,830 | Augusta Hotel
  [23] $167,980 | Hotel Boutique Casa Vander
  [24] OMITIDO (no es hotel): Hostal Miramar
  [25] $7

...

Hoteles encontrados: 25
  [1] $147,245 | Gran Hotel Vicente Costanera
  [2] $150,624 | ibis Puerto Montt
  [3] $278,217 | Courtyard by Marriott Puerto Montt
  [4] $113,737 | Hotel Vista Mar
  [5] $149,485 | Hotel Gran Pacifico
  [6] $204,800 | Novotel Puerto Montt
  [7] $69,992 | Hotel Angelmontt
  [8] $82,888 | Hotel Gran Luna
  [9] $88,190 | Hotel Le Mirage
  [10] $131,234 | Hotel Antupiren
  [11] $206,476 | Hotel Diego de Almagro Puerto Montt
  [12] OMITIDO (no es hotel): Hotel Apart Colón
  [13] $141,733 | Hotel Seminario
  [14] $92,162 | Casona Alemana
  [15] $57,743 | HOTEL NUNA
  [16] $162,731 | abba Presidente Suites Puerto Montt
  [17] $111,777 | Hotel Costa del Mar
  [18] $96,239 | Complejo Turistico los Alamos
  [19] OMITIDO (no es hotel): Apart Hotel Croacia
  [20] $116,000 | Hotel Terrazas del Mar
  [21] $139,983 | Park Inn by Radisson Puerto Varas
  [22] OMITIDO (no es hotel): Hotel Cabaña Del Lago Puerto Varas
  [23] $143,430 | Hotel Germania
  [24] $314,963 | Radisson H

>>> Presiona ENTER para comenzar a extraer datos <<<
 


Hoteles encontrados: 25
  [1] $240,946 | Park Lake Luxury Hotel
  [2] $148,732 | Hotel Pucón Indómito
  [3] $142,407 | Newen B&B
  [4] $116,694 | Hotel Cumbres del Sur
  [5] $130,570 | Rangi Pucon
  [6] $126,143 | Hotel Inti Kuyen Plaza
  [7] $137,184 | Hotel Vientos del Sur
  [8] $150,220 | Hotel Pucon Green Park
  [9] OMITIDO (no es hotel): Magma Lodge, Pucon
  [10] $138,540 | Hotel Küyen-Ko Pucón
  [11] $265,732 | Hotel Enjoy Pucón
  [12] $111,987 | La Araucaria Hostería Boutique - Ex
  [13] $209,975 | Martina de Goñi
  [14] $459,321 | Hotel Antumalal
  [15] $157,481 | Hotel Geronimo
  [16] OMITIDO (no es hotel): Hotel & Apart Hotel Monte Verde
  [17] $115,000 | Espacio Ecole
  [18] $96,886 | Santa Maria Pucon
  [19] $126,000 | Hotel del Volcán
  [20] $201,244 | Hotel Posada del Río - Parque Metre
  [21] OMITIDO (no es hotel): Hostal Pucon Sur
  [22] $117,648 | Hotel los maitenes
  [23] OMITIDO (no es hotel): French Andes Apart Hostel
  [24] $150,000 | Loft Pucon
  [25] OMITIDO (no 

>>> Presiona ENTER para comenzar a extraer datos <<<
 


Hoteles encontrados: 25
  [1] $92,835 | Hotel Boutique Casa Werner
  [2] $139,983 | Park Inn by Radisson Puerto Varas
  [3] $143,430 | Hotel Germania
  [4] $211,140 | Hotel Patagonico
  [5] $208,225 | Solace Hotel Puerto Varas
  [6] $314,963 | Radisson Hotel Puerto Varas
  [7] OMITIDO (no es hotel): Hotel Cabaña Del Lago Puerto Varas
  [8] $416,171 | Casa Molino Hotel Boutique & Restau
  [9] $167,490 | Hotel Museo El Greco Puerto Varas
  [10] $330,711 | Wyndham Puerto Varas Pettra
  [11] $132,284 | Casa Kalfu Hotel Boutique
  [12] OMITIDO (no es hotel): Ampe Lodge
  [13] $126,685 | Hotel Agua Nativa
  [14] OMITIDO (no es hotel): Hotel y cabañas Terrazas del Lago
  [15] $208,225 | Hotel Puelche
  [16] $160,981 | Puerto Chico Hotel
  [17] $162,906 | Hotel Puerta del Lago
  [18] $104,113 | Dein Haus Hotel y Departamentos
  [19] $152,144 | Hotel Borde Lago
  [20] $301,183 | Hotel Bellavista
  [21] $136,000 | Hotel Boutique Casa&Alma
  [22] $135,784 | Hotel Casona Freire
  [23] $119,161 | C

In [6]:
print(coleccion.count_documents({'integrante': 'Camila-rojas'}))

195
